# Bronze Layer - CBBC Summary Ingestion
Load raw CBBC Summary Excel files from Azure Data Lake Storage into `bronze.cbbc_summary` and record processing results in `bronze.cbbc_summary_ingestion_log`.

In [0]:
from datetime import datetime

from pyspark.sql import functions as F
from pyspark.sql.functions import (
    monotonically_increasing_id,
    col,
    lit,
    to_date,
    row_number,
    current_timestamp
)
from pyspark.sql.window import Window

## 1. Configure storage access

In [0]:
# Configure OAuth access to ADLS
service_credential = dbutils.secrets.get(scope="cbbcscope", key="app-secret")

spark.conf.set("fs.azure.account.auth.type.cbbcstakedatalake.dfs.core.windows.net", "OAuth")
spark.conf.set("fs.azure.account.oauth.provider.type.cbbcstakedatalake.dfs.core.windows.net", "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider")
spark.conf.set("fs.azure.account.oauth2.client.id.cbbcstakedatalake.dfs.core.windows.net", "08535731-dcba-457d-a98c-45558d70b8dd")
spark.conf.set("fs.azure.account.oauth2.client.secret.cbbcstakedatalake.dfs.core.windows.net", service_credential)
spark.conf.set("fs.azure.account.oauth2.client.endpoint.cbbcstakedatalake.dfs.core.windows.net", "https://login.windows.net/8b69ada7-612e-4db3-9499-2b3396adf7e0/oauth2/token")

## 2. Identify new source files

In [0]:
# List incoming Excel files from the source folder
raw_cbbc_summary = "abfss://source@cbbcstakedatalake.dfs.core.windows.net/cbbc_summary/"

files = dbutils.fs.ls(raw_cbbc_summary)

incoming_df = spark.createDataFrame(
    [(f.name, f.path) for f in files if f.path.endswith(".xlsx")],
    ["file_name", "file_path"]
)

# Get files that have not already been processed successfully
processed_df = (
    spark.table("bronze.cbbc_summary_ingestion_log")
         .filter(F.col("status") == "SUCCESS")
         .select("file_name")
         .distinct()
)

new_files_df = incoming_df.join(processed_df, on="file_name", how="left_anti")

new_files = [
    {
        "file_name": r["file_name"],
        "file_path": r["file_path"]
    }
    for r in new_files_df.collect()
]

## 3. Process each Excel file and write ingestion log

In [0]:
# Process each Excel file
for file in new_files:
    print(file["file_name"])

    trade_date_str = None
    issuer = None
    status = None
    error_message = None

    try:
        # Read Excel file
        excel_path = file["file_path"]

        df = (
            spark.read
                 .format("excel")
                 .load(excel_path)
        )

        # Extract trade date from the worksheet
        date_DDMMYYYY = str(int(float(df.select("_c1").collect()[7][0])))
        date_DDMMYYYY = date_DDMMYYYY.zfill(8)
        trade_date_str = datetime.strptime(date_DDMMYYYY, "%d%m%Y").strftime("%Y-%m-%d")

        # Remove non-data header rows
        w = Window.orderBy(monotonically_increasing_id())

        df = (
            df.withColumn("_rn", row_number().over(w))
              .filter(col("_rn") > 11)
              .drop("_rn")
        )

        # Extract issuer code from the first data row
        issuer = str(df.select("_c0").collect()[0][0])[:2]

        print(trade_date_str, issuer)

        # Assign standard column names
        df = df.toDF(
            "stock_short_name",
            "stock_code",
            "contracts_bought_count",
            "average_price_per_contract_bought",
            "contracts_sold_count",
            "average_price_per_contract_sold",
            "contracts_outstanding_count",
            "total_issue_size",
            "percent_issue_outstanding",
            "trading_currency",
            "meets_active_quote_criteria",
        )

        # Add ingestion metadata
        df = df.withColumn("trade_date", to_date(lit(trade_date_str)))
        df = df.withColumn("processed_at", current_timestamp())

        (
            df.write
              .mode("append")
              .format("delta")
              .saveAsTable("bronze.cbbc_summary")
        )

        status = "SUCCESS"
        error_message = "-"
        print("SUCCESS")

    except Exception as e:
        print("FAILED")
        print(e)

        if not trade_date_str:
            trade_date_str = "-"

        if not issuer:
            issuer = "-"

        status = "FAILED"
        error_message = str(e)[:5000]

    # Write processing result to ingestion log
    log_df = spark.createDataFrame(
        [(file["file_name"], trade_date_str, issuer, status, error_message)],
        ["file_name", "trade_date", "issuer", "status", "error_message"]
    )

    log_df = log_df.withColumn("processed_at", current_timestamp())

    log_df.write.mode("append").saveAsTable("bronze.cbbc_summary_ingestion_log")